In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("games_details.csv",low_memory=False)

print(data.shape)

display(data.head())

(668628, 29)


,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,OREB,DREB,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS
0,22200477,1610612759,SAS,San Antonio,1629641,Romeo Langford,Romeo,F,NaN,18:06,...,1.0,1.0,2.0,0.0,1.0,0.0,2.0,5.0,2.0,-2.0
1,22200477,1610612759,SAS,San Antonio,1631110,Jeremy Sochan,Jeremy,F,NaN,31:01,...,6.0,3.0,9.0,6.0,1.0,0.0,2.0,1.0,23.0,-14.0
2,22200477,1610612759,SAS,San Antonio,1627751,Jakob Poeltl,Jakob,C,NaN,21:42,...,1.0,3.0,4.0,1.0,1.0,0.0,2.0,4.0,13.0,-4.0
3,22200477,1610612759,SAS,San Antonio,1630170,Devin Vassell,Devin,G,NaN,30:20,...,0.0,9.0,9.0,5.0,3.0,0.0,2.0,1.0,10.0,-18.0
4,22200477,1610612759,SAS,San Antonio,1630200,Tre Jones,Tre,G,NaN,27:44,...,0.0,2.0,2.0,3.0,0.0,0.0,2.0,2.0,19.0,0.0


In [2]:
print(data.columns.tolist())

['GAME_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_CITY', 'PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'START_POSITION', 'COMMENT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS']


In [3]:
print("Nombre de joueurs :", data["PLAYER_ID"].nunique())

print("Nombre de matchs :", data["GAME_ID"].nunique())

Nombre de joueurs : 2687
Nombre de matchs : 26523


In [4]:
# Transformer GAME_ID en texte
data["GAME_ID"] = data["GAME_ID"].astype(str).str.zfill(10)

# Garder uniquement les matchs de saison régulière
data_reg = data[data["GAME_ID"].str[2] == "2"].copy()

# Vérifier le résultat
print("Matchs avant filtrage :", data["GAME_ID"].nunique())
print("Matchs après filtrage :", data_reg["GAME_ID"].nunique())

Matchs avant filtrage : 26523
Matchs après filtrage : 23244


In [5]:
print(data_reg["GAME_ID"].str[2].value_counts())

GAME_ID
2    578274
Name: count, dtype: int64


In [6]:
games = pd.read_csv("games.csv", low_memory=False)

games["GAME_ID"] = games["GAME_ID"].astype(str).str.zfill(10)

saisons = games[["GAME_ID", "SEASON"]].drop_duplicates()

display(saisons.head())

,GAME_ID,SEASON
0,0022200477,2022
1,0022200478,2022
2,0022200466,2022
3,0022200467,2022
4,0022200468,2022


In [7]:
print("Nombre de GAME_ID dupliqués :", saisons["GAME_ID"].duplicated().sum())

Nombre de GAME_ID dupliqués : 0


In [8]:
data_reg = data_reg.merge(
    saisons,
    on="GAME_ID",
    how="left",
    validate="many_to_one"
)

display(data_reg[["PLAYER_NAME", "GAME_ID", "SEASON", "PTS"]].head())

,PLAYER_NAME,GAME_ID,SEASON,PTS
0,Romeo Langford,0022200477,2022,2.0
1,Jeremy Sochan,0022200477,2022,23.0
2,Jakob Poeltl,0022200477,2022,13.0
3,Devin Vassell,0022200477,2022,10.0
4,Tre Jones,0022200477,2022,19.0


In [9]:
print("Nombre de lignes :", len(data_reg))
print("Saisons manquantes :", data_reg["SEASON"].isna().sum())

Nombre de lignes : 578274
Saisons manquantes : 0


In [10]:
moyennes = data_reg.groupby(
    ["PLAYER_ID", "SEASON"]
)["PTS"].mean().reset_index()

display(moyennes.head(10))

,PLAYER_ID,SEASON,PTS
0,15,2003,4.102041
1,15,2004,4.764706
2,15,2005,2.034483
3,15,2006,2.454545
4,15,2007,2.437500
5,42,2003,NaN
6,43,2003,3.133333
7,56,2003,14.621951
8,56,2004,11.337662
9,56,2005,7.728395


In [11]:
stats = [
    "PTS", "REB", "AST", "STL", "BLK",
    "FGA", "FGM", "FG3A", "FG3M",
    "FTA", "FTM", "OREB", "DREB",
    "TO", "PF"
]

moyennes_stats = data_reg.groupby(
    ["PLAYER_ID", "SEASON"]
)[stats].mean().reset_index()

display(moyennes_stats.head())

,PLAYER_ID,SEASON,PTS,REB,AST,STL,BLK,FGA,FGM,FG3A,FG3M,FTA,FTM,OREB,DREB,TO,PF
0,15,2003,4.102041,1.489796,0.530612,0.326531,0.102041,3.897959,1.469388,2.489796,0.877551,0.326531,0.285714,0.204082,1.285714,0.591837,0.857143
1,15,2004,4.764706,1.161765,0.750000,0.426471,0.014706,3.794118,1.632353,1.970588,0.838235,0.823529,0.661765,0.220588,0.941176,0.544118,0.985294
2,15,2005,2.034483,0.793103,0.448276,0.206897,0.034483,2.103448,0.827586,1.137931,0.310345,0.172414,0.068966,0.137931,0.655172,0.379310,0.689655
3,15,2006,2.454545,0.818182,0.363636,0.000000,0.090909,2.272727,0.818182,1.636364,0.636364,0.181818,0.181818,0.181818,0.636364,0.454545,0.545455
4,15,2007,2.437500,0.750000,0.625000,0.000000,0.062500,2.062500,0.750000,1.625000,0.687500,0.250000,0.250000,0.062500,0.687500,0.375000,0.687500


In [12]:
totaux = data_reg.groupby(
    ["PLAYER_ID", "SEASON"]
)[["FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA"]].sum().reset_index()

display(totaux.head(10))

,PLAYER_ID,SEASON,FGM,FGA,FG3M,FG3A,FTM,FTA
0,15,2003,72.0,191.0,43.0,122.0,14.0,16.0
1,15,2004,111.0,258.0,57.0,134.0,45.0,56.0
2,15,2005,24.0,61.0,9.0,33.0,2.0,5.0
3,15,2006,9.0,25.0,7.0,18.0,2.0,2.0
4,15,2007,12.0,33.0,11.0,26.0,4.0,4.0
5,42,2003,0.0,0.0,0.0,0.0,0.0,0.0
6,43,2003,17.0,42.0,8.0,17.0,5.0,5.0
7,56,2003,482.0,1024.0,55.0,165.0,180.0,252.0
8,56,2004,339.0,725.0,42.0,129.0,153.0,201.0
9,56,2005,230.0,547.0,66.0,230.0,100.0,126.0


In [13]:
totaux["FG_PCT"] = totaux["FGM"] / totaux["FGA"] * 100

totaux["FG3_PCT"] = totaux["FG3M"] / totaux["FG3A"] * 100

totaux["FT_PCT"] = totaux["FTM"] / totaux["FTA"] * 100

display(totaux.head(10))

,PLAYER_ID,SEASON,FGM,FGA,FG3M,FG3A,FTM,FTA,FG_PCT,FG3_PCT,FT_PCT
0,15,2003,72.0,191.0,43.0,122.0,14.0,16.0,37.696335,35.245902,87.500000
1,15,2004,111.0,258.0,57.0,134.0,45.0,56.0,43.023256,42.537313,80.357143
2,15,2005,24.0,61.0,9.0,33.0,2.0,5.0,39.344262,27.272727,40.000000
3,15,2006,9.0,25.0,7.0,18.0,2.0,2.0,36.000000,38.888889,100.000000
4,15,2007,12.0,33.0,11.0,26.0,4.0,4.0,36.363636,42.307692,100.000000
5,42,2003,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
6,43,2003,17.0,42.0,8.0,17.0,5.0,5.0,40.476190,47.058824,100.000000
7,56,2003,482.0,1024.0,55.0,165.0,180.0,252.0,47.070312,33.333333,71.428571
8,56,2004,339.0,725.0,42.0,129.0,153.0,201.0,46.758621,32.558140,76.119403
9,56,2005,230.0,547.0,66.0,230.0,100.0,126.0,42.047532,28.695652,79.365079


In [14]:
colonnes = [
    "REB", "AST", "STL", "BLK",
    "FGA", "FGM", "FG3A", "FG3M",
    "FTA", "FTM", "OREB", "DREB",
    "TO", "PF"
]

moyennes = moyennes.merge(
    moyennes_stats[["PLAYER_ID", "SEASON"] + colonnes],
    on=["PLAYER_ID", "SEASON"],
    how="left",
    validate="one_to_one"
)

display(moyennes.head(10))

,PLAYER_ID,SEASON,PTS,REB,AST,STL,BLK,FGA,FGM,FG3A,FG3M,FTA,FTM,OREB,DREB,TO,PF
0,15,2003,4.102041,1.489796,0.530612,0.326531,0.102041,3.897959,1.469388,2.489796,0.877551,0.326531,0.285714,0.204082,1.285714,0.591837,0.857143
1,15,2004,4.764706,1.161765,0.750000,0.426471,0.014706,3.794118,1.632353,1.970588,0.838235,0.823529,0.661765,0.220588,0.941176,0.544118,0.985294
2,15,2005,2.034483,0.793103,0.448276,0.206897,0.034483,2.103448,0.827586,1.137931,0.310345,0.172414,0.068966,0.137931,0.655172,0.379310,0.689655
3,15,2006,2.454545,0.818182,0.363636,0.000000,0.090909,2.272727,0.818182,1.636364,0.636364,0.181818,0.181818,0.181818,0.636364,0.454545,0.545455
4,15,2007,2.437500,0.750000,0.625000,0.000000,0.062500,2.062500,0.750000,1.625000,0.687500,0.250000,0.250000,0.062500,0.687500,0.375000,0.687500
5,42,2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,43,2003,3.133333,0.866667,0.933333,0.400000,0.133333,2.800000,1.133333,1.133333,0.533333,0.333333,0.333333,0.200000,0.666667,0.400000,0.400000
7,56,2003,14.621951,4.170732,5.475610,1.170732,0.231707,12.487805,5.878049,2.012195,0.670732,3.073171,2.195122,0.878049,3.292683,1.841463,2.060976
8,56,2004,11.337662,3.064935,6.090909,1.142857,0.155844,9.415584,4.402597,1.675325,0.545455,2.610390,1.987013,0.623377,2.441558,1.922078,1.571429
9,56,2005,7.728395,2.876543,3.172840,0.876543,0.123457,6.753086,2.839506,2.839506,0.814815,1.555556,1.234568,0.419753,2.456790,1.259259,2.086420


In [15]:
print(moyennes.columns.tolist())

['PLAYER_ID', 'SEASON', 'PTS', 'REB', 'AST', 'STL', 'BLK', 'FGA', 'FGM', 'FG3A', 'FG3M', 'FTA', 'FTM', 'OREB', 'DREB', 'TO', 'PF']


In [17]:
doublons = moyennes[
    moyennes.duplicated(
        subset=["PLAYER_ID", "SEASON"],
        keep=False
    )
]

display(doublons[["PLAYER_ID", "SEASON"]].head(20))

,PLAYER_ID,SEASON


In [22]:
pourcentages = totaux[
    ["PLAYER_ID", "SEASON", "FG_PCT", "FG3_PCT", "FT_PCT"]
]

moyennes = moyennes.merge(
    pourcentages,
    on=["PLAYER_ID", "SEASON"],
    how="left",
    validate="one_to_one"
)

display(moyennes.head(10))

,PLAYER_ID,SEASON,PTS,FG_PCT,FG3_PCT,FT_PCT
0,15,2003,4.102041,37.696335,35.245902,87.500000
1,15,2004,4.764706,43.023256,42.537313,80.357143
2,15,2005,2.034483,39.344262,27.272727,40.000000
3,15,2006,2.454545,36.000000,38.888889,100.000000
4,15,2007,2.437500,36.363636,42.307692,100.000000
5,42,2003,NaN,NaN,NaN,NaN
6,43,2003,3.133333,40.476190,47.058824,100.000000
7,56,2003,14.621951,47.070312,33.333333,71.428571
8,56,2004,11.337662,46.758621,32.558140,76.119403
9,56,2005,7.728395,42.047532,28.695652,79.365079


In [18]:
noms = data_reg.groupby("PLAYER_ID")["PLAYER_NAME"].last().reset_index()

display(noms.head(10))

,PLAYER_ID,PLAYER_NAME
0,15,Eric Piatkowski
1,42,Monty Williams
2,43,Chris Whitney
3,56,Gary Payton
4,57,Doug Christie
5,72,Kenny Anderson
6,77,Doug Overton
7,84,Latrell Sprewell
8,87,Dikembe Mutombo
9,89,Nick Van Exel


In [19]:
print(data_reg["MIN"].head(10))

0    18:06
1    31:01
2    21:42
3    30:20
4    27:44
5    18:04
6    16:55
7    21:20
8    21:10
9     9:16
Name: MIN, dtype: object


In [20]:
def convertir_minutes(temps):
    if pd.isna(temps):
        return np.nan

    minutes, secondes = temps.split(":")

    return float(minutes) + float(secondes) / 60

In [21]:
data_reg["MIN_DEC"] = data_reg["MIN"].apply(convertir_minutes)

display(data_reg[["PLAYER_NAME","SEASON", "MIN", "MIN_DEC"]].head(10))

,PLAYER_NAME,SEASON,MIN,MIN_DEC
0,Romeo Langford,2022,18:06,18.100000
1,Jeremy Sochan,2022,31:01,31.016667
2,Jakob Poeltl,2022,21:42,21.700000
3,Devin Vassell,2022,30:20,30.333333
4,Tre Jones,2022,27:44,27.733333
5,Zach Collins,2022,18:04,18.066667
6,Doug McDermott,2022,16:55,16.916667
7,Josh Richardson,2022,21:20,21.333333
8,Malaki Branham,2022,21:10,21.166667
9,Keita Bates-Diop,2022,9:16,9.266667


In [22]:
minutes_moyennes = data_reg.groupby(
    ["PLAYER_ID", "SEASON"]
)["MIN_DEC"].mean().reset_index()

display(minutes_moyennes.head(10))

,PLAYER_ID,SEASON,MIN_DEC
0,15,2003,14.328231
1,15,2004,12.340931
2,15,2005,7.870115
3,15,2006,6.618182
4,15,2007,7.041667
5,42,2003,NaN
6,43,2003,11.706667
7,56,2003,34.586179
8,56,2004,33.125325
9,56,2005,28.455761


In [23]:
moyennes = moyennes.merge(
    minutes_moyennes,
    on=["PLAYER_ID", "SEASON"],
    how="left",
    validate="one_to_one"
)

display(moyennes[["PLAYER_ID", "SEASON", "PTS", "REB", "AST", "MIN_DEC"]].head(10))

,PLAYER_ID,SEASON,PTS,REB,AST,MIN_DEC
0,15,2003,4.102041,1.489796,0.530612,14.328231
1,15,2004,4.764706,1.161765,0.750000,12.340931
2,15,2005,2.034483,0.793103,0.448276,7.870115
3,15,2006,2.454545,0.818182,0.363636,6.618182
4,15,2007,2.437500,0.750000,0.625000,7.041667
5,42,2003,NaN,NaN,NaN,NaN
6,43,2003,3.133333,0.866667,0.933333,11.706667
7,56,2003,14.621951,4.170732,5.475610,34.586179
8,56,2004,11.337662,3.064935,6.090909,33.125325
9,56,2005,7.728395,2.876543,3.172840,28.455761


In [24]:
noms = data_reg.groupby("PLAYER_ID")["PLAYER_NAME"].last().reset_index()

display(noms.head(10))

,PLAYER_ID,PLAYER_NAME
0,15,Eric Piatkowski
1,42,Monty Williams
2,43,Chris Whitney
3,56,Gary Payton
4,57,Doug Christie
5,72,Kenny Anderson
6,77,Doug Overton
7,84,Latrell Sprewell
8,87,Dikembe Mutombo
9,89,Nick Van Exel


In [25]:
moyennes = moyennes.merge(
    noms,
    on="PLAYER_ID",
    how="left",
    validate="many_to_one"
)
display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "PTS", "REB", "AST", "MIN_DEC"]
    ].head(10)
)

,PLAYER_NAME,SEASON,PTS,REB,AST,MIN_DEC
0,Eric Piatkowski,2003,4.102041,1.489796,0.530612,14.328231
1,Eric Piatkowski,2004,4.764706,1.161765,0.750000,12.340931
2,Eric Piatkowski,2005,2.034483,0.793103,0.448276,7.870115
3,Eric Piatkowski,2006,2.454545,0.818182,0.363636,6.618182
4,Eric Piatkowski,2007,2.437500,0.750000,0.625000,7.041667
5,Monty Williams,2003,NaN,NaN,NaN,NaN
6,Chris Whitney,2003,3.133333,0.866667,0.933333,11.706667
7,Gary Payton,2003,14.621951,4.170732,5.475610,34.586179
8,Gary Payton,2004,11.337662,3.064935,6.090909,33.125325
9,Gary Payton,2005,7.728395,2.876543,3.172840,28.455761


In [26]:
data_reg["JOUE"] = data_reg["MIN_DEC"].notna()

display(data_reg[["PLAYER_NAME", "MIN", "JOUE"]].head(10))

,PLAYER_NAME,MIN,JOUE
0,Romeo Langford,18:06,True
1,Jeremy Sochan,31:01,True
2,Jakob Poeltl,21:42,True
3,Devin Vassell,30:20,True
4,Tre Jones,27:44,True
5,Zach Collins,18:04,True
6,Doug McDermott,16:55,True
7,Josh Richardson,21:20,True
8,Malaki Branham,21:10,True
9,Keita Bates-Diop,9:16,True


In [27]:
print(data_reg["JOUE"].value_counts())

display(data_reg[data_reg["JOUE"] == False].head(10))

JOUE
True     482739
False     95535
Name: count, dtype: int64


,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,SEASON,MIN_DEC,JOUE
13,0022200477,1610612759,SAS,San Antonio,203476,Gorgui Dieng,Gorgui,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022,NaN,False
51,0022200478,1610612762,UTA,Utah,1630695,Micah Potter,Micah,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022,NaN,False
62,0022200466,1610612749,MIL,Milwaukee,203648,Thanasis Antetokounmpo,Thanasis,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022,NaN,False
63,0022200466,1610612749,MIL,Milwaukee,201588,George Hill,George,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022,NaN,False
64,0022200466,1610612749,MIL,Milwaukee,201586,Serge Ibaka,Serge,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022,NaN,False
65,0022200466,1610612749,MIL,Milwaukee,1629670,Jordan Nwora,Jordan,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022,NaN,False
74,0022200466,1610612739,CLE,Cleveland,1629603,Mamadi Diakite,Mamadi,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022,NaN,False
75,0022200466,1610612739,CLE,Cleveland,201577,Robin Lopez,Robin,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022,NaN,False
76,0022200466,1610612739,CLE,Cleveland,203526,Raul Neto,Raul,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022,NaN,False
89,0022200467,1610612765,DET,Detroit,203457,Nerlens Noel,Nerlens,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022,NaN,False


In [29]:
print(data_reg[data_reg["JOUE"] == False]["COMMENT"].value_counts().head(35))

COMMENT
DNP - Coach's Decision                      52428
DNP - Coach's Decision                      24671
DND - Injury/Illness                          980
DND - Injury/Illness                          833
DNP - Injury/Illness                          810
DNP - Injury/Illness                          701
DND - Sprained Left Ankle                     249
DND - Sprained Right Ankle                    185
NWT - Personal Reasons                        163
NWT - Injury/Illness                          155
DND - Rest                                    134
DND - Left Ankle Sprain                       124
DND - Back Spasms                             122
DND - Sore Left Knee                          114
NWT - NBA Suspension                          108
DND - Sore Right Knee                         103
DND - Sore Lower Back                          92
NWT - Personal                                 87
DND - Right Ankle Sprain                       82
DND - Coach's Decision                    

In [30]:
sans_minutes = data_reg[data_reg["JOUE"] == False]

print("Lignes sans minutes :", len(sans_minutes))

print("Sans minutes et sans commentaire :",
      sans_minutes["COMMENT"].isna().sum())

Lignes sans minutes : 95535
Sans minutes et sans commentaire : 0


In [31]:
matchs = data_reg.groupby(
    ["PLAYER_ID", "SEASON"]
)["JOUE"].sum().reset_index()

matchs = matchs.rename(columns={"JOUE": "GP"})

display(matchs.head(10))

,PLAYER_ID,SEASON,GP
0,15,2003,49
1,15,2004,68
2,15,2005,29
3,15,2006,11
4,15,2007,16
5,42,2003,0
6,43,2003,15
7,56,2003,82
8,56,2004,77
9,56,2005,81


In [32]:
print(data_reg["START_POSITION"].value_counts(dropna=False))

START_POSITION
NaN    345719
G       93031
F       93025
C       46499
Name: count, dtype: int64


In [33]:
titulaires = data_reg.groupby("GAME_ID")["START_POSITION"].count()

print(titulaires.value_counts().sort_index())

START_POSITION
9         6
10    23178
11       25
12       19
13       10
14        3
15        2
16        1
Name: count, dtype: int64


In [34]:
doublons = data_reg.duplicated(
    subset=["GAME_ID", "PLAYER_ID"],
    keep=False
)

print("Nombre de lignes dupliquées :", doublons.sum())

display(
    data_reg.loc[
        doublons,
        ["GAME_ID", "PLAYER_NAME", "START_POSITION"]
    ].head(20)
)

Nombre de lignes dupliquées : 578


,GAME_ID,PLAYER_NAME,START_POSITION
55727,0022000069,Kyle Anderson,F
55731,0022000069,Desmond Bane,NaN
55733,0022000069,Sean McDermott,NaN
55735,0022000069,Grayson Allen,NaN
55736,0022000069,Jontay Porter,NaN
55737,0022000069,Xavier Tillman,NaN
55741,0022000069,Terry Rozier,G
55749,0022000069,Vernon Carey Jr.,NaN
55750,0022000069,Malik Monk,NaN
55751,0022000070,Jayson Tatum,F


In [35]:
display(
    data_reg[
        (data_reg["GAME_ID"] == "0022000069") &
        (data_reg["PLAYER_NAME"] == "Kyle Anderson")
    ][["GAME_ID", "PLAYER_NAME", "MIN", "PTS", "REB", "AST"]]
)

,GAME_ID,PLAYER_NAME,MIN,PTS,REB,AST
55727,0022000069,Kyle Anderson,29:10,18.0,11.0,5.0
72402,0022000069,Kyle Anderson,29:10,18.0,11.0,5.0


In [36]:
print("Doublons exacts :", data_reg.duplicated().sum())

print("Doublons joueur-match :",
      data_reg.duplicated(subset=["GAME_ID", "PLAYER_ID"]).sum())

Doublons exacts : 0
Doublons joueur-match : 289


In [37]:
lignes = data_reg[
    (data_reg["GAME_ID"] == "0022000069") &
    (data_reg["PLAYER_NAME"] == "Kyle Anderson")
]

display(lignes.T)

,55727,72402
GAME_ID,0022000069,0022000069
TEAM_ID,1610612763,1610612763
TEAM_ABBREVIATION,MEM,MEM
TEAM_CITY,Memphis,Memphis
PLAYER_ID,203937,203937
PLAYER_NAME,Kyle Anderson,Kyle Anderson
NICKNAME,NaN,NaN
START_POSITION,F,F
COMMENT,NaN,NaN
MIN,29:10,29:10


In [38]:
print(lignes.iloc[0].equals(lignes.iloc[1]))

False


In [39]:
ligne1 = lignes.iloc[0]
ligne2 = lignes.iloc[1]

differences = []

for colonne in lignes.columns:
    if not (
        ligne1[colonne] == ligne2[colonne]
        or (pd.isna(ligne1[colonne]) and pd.isna(ligne2[colonne]))
    ):
        differences.append(colonne)

print("Colonnes différentes :", differences)

Colonnes différentes : ['FG3_PCT']


In [40]:
print("Ligne 1 :", repr(lignes.iloc[0]["FG3_PCT"]))
print("Ligne 2 :", repr(lignes.iloc[1]["FG3_PCT"]))

Ligne 1 : np.float64(0.333)
Ligne 2 : np.float64(0.3329999999999999)


In [41]:
colonnes = [
    "GAME_ID", "PLAYER_ID", "MIN",
    "FGM", "FGA", "FG3M", "FG3A",
    "FTM", "FTA", "OREB", "DREB",
    "REB", "AST", "STL", "BLK",
    "TO", "PF", "PTS", "START_POSITION"
]

print(
    "Doublons sur les statistiques :",
    data_reg.duplicated(subset=colonnes).sum()
)

Doublons sur les statistiques : 283


In [42]:
doublons_differents = data_reg[
    data_reg.duplicated(
        subset=["GAME_ID", "PLAYER_ID"], 
        keep=False
    )
]

doublons_differents = doublons_differents[
    ~doublons_differents.duplicated(
        subset=colonnes,
        keep=False
    )
]

display(
    doublons_differents[
        ["GAME_ID", "PLAYER_NAME", "MIN", "PTS",
         "REB", "AST", "FGM", "FGA", "FG3M", "FG3A"]
    ]
)

,GAME_ID,PLAYER_NAME,MIN,PTS,REB,AST,FGM,FGA,FG3M,FG3A
55754,0022000070,Jaylen Brown,37:34,25.0,9.0,2.0,12.0,20.0,1.0,7.0
55756,0022000070,Payton Pritchard,16:00,0.0,0.0,2.0,0.0,3.0,0.0,3.0
55757,0022000070,Grant Williams,19:23,8.0,5.0,0.0,4.0,7.0,0.0,1.0
55766,0022000070,Jerami Grant,37:26,24.0,4.0,0.0,9.0,24.0,2.0,9.0
55769,0022000070,Killian Hayes,23:24,0.0,1.0,6.0,0.0,5.0,0.0,2.0
55771,0022000070,Isaiah Stewart,18:24,8.0,5.0,0.0,3.0,5.0,0.0,0.0
55814,0022000070,Jaylen Brown,37:34,25.0,9.0,2.0,12.0,21.0,1.0,7.0
55815,0022000070,Payton Pritchard,16:00,0.0,0.0,2.0,0.0,3.0,0.0,3.0
55816,0022000070,Grant Williams,19:23,8.0,4.0,0.0,4.0,6.0,0.0,1.0
55820,0022000070,Jerami Grant,37:26,24.0,5.0,0.0,9.0,24.0,2.0,9.0


In [43]:
display(
    data_reg[
        (data_reg["GAME_ID"] == "0022000070") &
        (data_reg["PLAYER_NAME"] == "Jaylen Brown")
    ].T
)

,55754,55814
GAME_ID,0022000070,0022000070
TEAM_ID,1610612738,1610612738
TEAM_ABBREVIATION,BOS,BOS
TEAM_CITY,Boston,Boston
PLAYER_ID,1627759,1627759
PLAYER_NAME,Jaylen Brown,Jaylen Brown
NICKNAME,NaN,NaN
START_POSITION,G,G
COMMENT,NaN,NaN
MIN,37:34,37:34


In [44]:
data_reg = data_reg.drop_duplicates(
    subset=["GAME_ID", "PLAYER_ID"],
    keep="last"
)

In [45]:
data_reg = data_reg.drop_duplicates(
    subset=["GAME_ID", "PLAYER_ID"],
    keep="last"
)

print("Nombre de lignes :", len(data_reg))

print("Doublons restants :", data_reg.duplicated(
    subset=["GAME_ID", "PLAYER_ID"]
).sum())

Nombre de lignes : 577985
Doublons restants : 0


In [46]:
display(
    data_reg[
        (data_reg["GAME_ID"] == "0022000070") &
        (data_reg["PLAYER_NAME"] == "Jaylen Brown")
    ][["PLAYER_NAME", "FGA", "FGM", "FG_PCT"]]
)

,PLAYER_NAME,FGA,FGM,FG_PCT
55814,Jaylen Brown,21.0,12.0,0.571


In [47]:
moyennes = data_reg.groupby(
    ["PLAYER_ID", "SEASON"]
)[stats].mean().reset_index()

display(moyennes.head(10))

,PLAYER_ID,SEASON,PTS,REB,AST,STL,BLK,FGA,FGM,FG3A,FG3M,FTA,FTM,OREB,DREB,TO,PF
0,15,2003,4.102041,1.489796,0.530612,0.326531,0.102041,3.897959,1.469388,2.489796,0.877551,0.326531,0.285714,0.204082,1.285714,0.591837,0.857143
1,15,2004,4.764706,1.161765,0.750000,0.426471,0.014706,3.794118,1.632353,1.970588,0.838235,0.823529,0.661765,0.220588,0.941176,0.544118,0.985294
2,15,2005,2.034483,0.793103,0.448276,0.206897,0.034483,2.103448,0.827586,1.137931,0.310345,0.172414,0.068966,0.137931,0.655172,0.379310,0.689655
3,15,2006,2.454545,0.818182,0.363636,0.000000,0.090909,2.272727,0.818182,1.636364,0.636364,0.181818,0.181818,0.181818,0.636364,0.454545,0.545455
4,15,2007,2.437500,0.750000,0.625000,0.000000,0.062500,2.062500,0.750000,1.625000,0.687500,0.250000,0.250000,0.062500,0.687500,0.375000,0.687500
5,42,2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,43,2003,3.133333,0.866667,0.933333,0.400000,0.133333,2.800000,1.133333,1.133333,0.533333,0.333333,0.333333,0.200000,0.666667,0.400000,0.400000
7,56,2003,14.621951,4.170732,5.475610,1.170732,0.231707,12.487805,5.878049,2.012195,0.670732,3.073171,2.195122,0.878049,3.292683,1.841463,2.060976
8,56,2004,11.337662,3.064935,6.090909,1.142857,0.155844,9.415584,4.402597,1.675325,0.545455,2.610390,1.987013,0.623377,2.441558,1.922078,1.571429
9,56,2005,7.728395,2.876543,3.172840,0.876543,0.123457,6.753086,2.839506,2.839506,0.814815,1.555556,1.234568,0.419753,2.456790,1.259259,2.086420


In [48]:
totaux = data_reg.groupby(
    ["PLAYER_ID", "SEASON"]
)[["FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA"]].sum()

totaux["FG_PCT"] = 100 * totaux["FGM"] / totaux["FGA"]
totaux["FG3_PCT"] = 100 * totaux["FG3M"] / totaux["FG3A"]
totaux["FT_PCT"] = 100 * totaux["FTM"] / totaux["FTA"]

pourcentages = totaux[
    ["FG_PCT", "FG3_PCT", "FT_PCT"]
].reset_index()

display(pourcentages.head(10))

,PLAYER_ID,SEASON,FG_PCT,FG3_PCT,FT_PCT
0,15,2003,37.696335,35.245902,87.500000
1,15,2004,43.023256,42.537313,80.357143
2,15,2005,39.344262,27.272727,40.000000
3,15,2006,36.000000,38.888889,100.000000
4,15,2007,36.363636,42.307692,100.000000
5,42,2003,NaN,NaN,NaN
6,43,2003,40.476190,47.058824,100.000000
7,56,2003,47.070312,33.333333,71.428571
8,56,2004,46.758621,32.558140,76.119403
9,56,2005,42.047532,28.695652,79.365079


In [49]:
moyennes = moyennes.merge(
    pourcentages,
    on=["PLAYER_ID", "SEASON"],
    how="left"
)

display(moyennes.head(10))

,PLAYER_ID,SEASON,PTS,REB,AST,STL,BLK,FGA,FGM,FG3A,FG3M,FTA,FTM,OREB,DREB,TO,PF,FG_PCT,FG3_PCT,FT_PCT
0,15,2003,4.102041,1.489796,0.530612,0.326531,0.102041,3.897959,1.469388,2.489796,0.877551,0.326531,0.285714,0.204082,1.285714,0.591837,0.857143,37.696335,35.245902,87.500000
1,15,2004,4.764706,1.161765,0.750000,0.426471,0.014706,3.794118,1.632353,1.970588,0.838235,0.823529,0.661765,0.220588,0.941176,0.544118,0.985294,43.023256,42.537313,80.357143
2,15,2005,2.034483,0.793103,0.448276,0.206897,0.034483,2.103448,0.827586,1.137931,0.310345,0.172414,0.068966,0.137931,0.655172,0.379310,0.689655,39.344262,27.272727,40.000000
3,15,2006,2.454545,0.818182,0.363636,0.000000,0.090909,2.272727,0.818182,1.636364,0.636364,0.181818,0.181818,0.181818,0.636364,0.454545,0.545455,36.000000,38.888889,100.000000
4,15,2007,2.437500,0.750000,0.625000,0.000000,0.062500,2.062500,0.750000,1.625000,0.687500,0.250000,0.250000,0.062500,0.687500,0.375000,0.687500,36.363636,42.307692,100.000000
5,42,2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,43,2003,3.133333,0.866667,0.933333,0.400000,0.133333,2.800000,1.133333,1.133333,0.533333,0.333333,0.333333,0.200000,0.666667,0.400000,0.400000,40.476190,47.058824,100.000000
7,56,2003,14.621951,4.170732,5.475610,1.170732,0.231707,12.487805,5.878049,2.012195,0.670732,3.073171,2.195122,0.878049,3.292683,1.841463,2.060976,47.070312,33.333333,71.428571
8,56,2004,11.337662,3.064935,6.090909,1.142857,0.155844,9.415584,4.402597,1.675325,0.545455,2.610390,1.987013,0.623377,2.441558,1.922078,1.571429,46.758621,32.558140,76.119403
9,56,2005,7.728395,2.876543,3.172840,0.876543,0.123457,6.753086,2.839506,2.839506,0.814815,1.555556,1.234568,0.419753,2.456790,1.259259,2.086420,42.047532,28.695652,79.365079


In [50]:
minutes_moyennes = data_reg.groupby(
    ["PLAYER_ID", "SEASON"]
)["MIN_DEC"].mean().reset_index()

moyennes = moyennes.merge(
    minutes_moyennes,
    on=["PLAYER_ID", "SEASON"],
    how="left"
)

display(moyennes.head(10))

,PLAYER_ID,SEASON,PTS,REB,AST,STL,BLK,FGA,FGM,FG3A,...,FTA,FTM,OREB,DREB,TO,PF,FG_PCT,FG3_PCT,FT_PCT,MIN_DEC
0,15,2003,4.102041,1.489796,0.530612,0.326531,0.102041,3.897959,1.469388,2.489796,...,0.326531,0.285714,0.204082,1.285714,0.591837,0.857143,37.696335,35.245902,87.500000,14.328231
1,15,2004,4.764706,1.161765,0.750000,0.426471,0.014706,3.794118,1.632353,1.970588,...,0.823529,0.661765,0.220588,0.941176,0.544118,0.985294,43.023256,42.537313,80.357143,12.340931
2,15,2005,2.034483,0.793103,0.448276,0.206897,0.034483,2.103448,0.827586,1.137931,...,0.172414,0.068966,0.137931,0.655172,0.379310,0.689655,39.344262,27.272727,40.000000,7.870115
3,15,2006,2.454545,0.818182,0.363636,0.000000,0.090909,2.272727,0.818182,1.636364,...,0.181818,0.181818,0.181818,0.636364,0.454545,0.545455,36.000000,38.888889,100.000000,6.618182
4,15,2007,2.437500,0.750000,0.625000,0.000000,0.062500,2.062500,0.750000,1.625000,...,0.250000,0.250000,0.062500,0.687500,0.375000,0.687500,36.363636,42.307692,100.000000,7.041667
5,42,2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,43,2003,3.133333,0.866667,0.933333,0.400000,0.133333,2.800000,1.133333,1.133333,...,0.333333,0.333333,0.200000,0.666667,0.400000,0.400000,40.476190,47.058824,100.000000,11.706667
7,56,2003,14.621951,4.170732,5.475610,1.170732,0.231707,12.487805,5.878049,2.012195,...,3.073171,2.195122,0.878049,3.292683,1.841463,2.060976,47.070312,33.333333,71.428571,34.586179
8,56,2004,11.337662,3.064935,6.090909,1.142857,0.155844,9.415584,4.402597,1.675325,...,2.610390,1.987013,0.623377,2.441558,1.922078,1.571429,46.758621,32.558140,76.119403,33.125325
9,56,2005,7.728395,2.876543,3.172840,0.876543,0.123457,6.753086,2.839506,2.839506,...,1.555556,1.234568,0.419753,2.456790,1.259259,2.086420,42.047532,28.695652,79.365079,28.455761


In [51]:
noms = data_reg.groupby("PLAYER_ID")["PLAYER_NAME"].last().reset_index()

moyennes = moyennes.merge(
    noms,
    on="PLAYER_ID",
    how="left"
)

display(moyennes[["PLAYER_ID", "PLAYER_NAME", "SEASON", "PTS", "REB", "AST", "MIN_DEC"]].head(10))

,PLAYER_ID,PLAYER_NAME,SEASON,PTS,REB,AST,MIN_DEC
0,15,Eric Piatkowski,2003,4.102041,1.489796,0.530612,14.328231
1,15,Eric Piatkowski,2004,4.764706,1.161765,0.750000,12.340931
2,15,Eric Piatkowski,2005,2.034483,0.793103,0.448276,7.870115
3,15,Eric Piatkowski,2006,2.454545,0.818182,0.363636,6.618182
4,15,Eric Piatkowski,2007,2.437500,0.750000,0.625000,7.041667
5,42,Monty Williams,2003,NaN,NaN,NaN,NaN
6,43,Chris Whitney,2003,3.133333,0.866667,0.933333,11.706667
7,56,Gary Payton,2003,14.621951,4.170732,5.475610,34.586179
8,56,Gary Payton,2004,11.337662,3.064935,6.090909,33.125325
9,56,Gary Payton,2005,7.728395,2.876543,3.172840,28.455761


In [52]:
matchs = data_reg.groupby(
    ["PLAYER_ID", "SEASON"]
)["JOUE"].sum().reset_index()

matchs = matchs.rename(columns={"JOUE": "GP"})

display(matchs.head(10))

,PLAYER_ID,SEASON,GP
0,15,2003,49
1,15,2004,68
2,15,2005,29
3,15,2006,11
4,15,2007,16
5,42,2003,0
6,43,2003,15
7,56,2003,82
8,56,2004,77
9,56,2005,81


In [53]:
data_reg["TITULAIRE"] = data_reg["START_POSITION"].notna()

titularisations = data_reg.groupby(
    ["PLAYER_ID", "SEASON"]
)["TITULAIRE"].sum().reset_index()

titularisations = titularisations.rename(
    columns={"TITULAIRE": "GS"}
)

display(titularisations.head(10))

,PLAYER_ID,SEASON,GS
0,15,2003,0
1,15,2004,11
2,15,2005,1
3,15,2006,0
4,15,2007,0
5,42,2003,0
6,43,2003,5
7,56,2003,82
8,56,2004,77
9,56,2005,25


In [54]:
verification = matchs.merge(
    titularisations,
    on=["PLAYER_ID", "SEASON"],
    how="left"
)

erreurs = verification[verification["GS"] > verification["GP"]]

print("Nombre d'anomalies :", len(erreurs))

display(erreurs.head(10))

Nombre d'anomalies : 0


,PLAYER_ID,SEASON,GP,GS


In [55]:
moyennes = moyennes.merge(
    verification,
    on=["PLAYER_ID", "SEASON"],
    how="left"
)

display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "GP", "GS",
         "PTS", "REB", "AST", "MIN_DEC"]
    ].head(10)
)

,PLAYER_NAME,SEASON,GP,GS,PTS,REB,AST,MIN_DEC
0,Eric Piatkowski,2003,49,0,4.102041,1.489796,0.530612,14.328231
1,Eric Piatkowski,2004,68,11,4.764706,1.161765,0.750000,12.340931
2,Eric Piatkowski,2005,29,1,2.034483,0.793103,0.448276,7.870115
3,Eric Piatkowski,2006,11,0,2.454545,0.818182,0.363636,6.618182
4,Eric Piatkowski,2007,16,0,2.437500,0.750000,0.625000,7.041667
5,Monty Williams,2003,0,0,NaN,NaN,NaN,NaN
6,Chris Whitney,2003,15,5,3.133333,0.866667,0.933333,11.706667
7,Gary Payton,2003,82,82,14.621951,4.170732,5.475610,34.586179
8,Gary Payton,2004,77,77,11.337662,3.064935,6.090909,33.125325
9,Gary Payton,2005,81,25,7.728395,2.876543,3.172840,28.455761


In [56]:
moyennes = moyennes[moyennes["GP"] > 0].copy()

print("Nombre de lignes restantes :", len(moyennes))
print("Joueurs-saisons avec GP = 0 :", (moyennes["GP"] == 0).sum())

Nombre de lignes restantes : 9728
Joueurs-saisons avec GP = 0 : 0


In [57]:
def convertir_minutes(minutes):
    total_secondes = round(minutes * 60)
    m = total_secondes // 60
    s = total_secondes % 60
    return f"{m:02d}:{s:02d}"

moyennes["MIN_MOY"] = moyennes["MIN_DEC"].apply(convertir_minutes)

display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "GP", "GS", "MIN_MOY", "PTS", "REB", "AST"]
    ].head(10)
)

,PLAYER_NAME,SEASON,GP,GS,MIN_MOY,PTS,REB,AST
0,Eric Piatkowski,2003,49,0,14:20,4.102041,1.489796,0.530612
1,Eric Piatkowski,2004,68,11,12:20,4.764706,1.161765,0.750000
2,Eric Piatkowski,2005,29,1,07:52,2.034483,0.793103,0.448276
3,Eric Piatkowski,2006,11,0,06:37,2.454545,0.818182,0.363636
4,Eric Piatkowski,2007,16,0,07:02,2.437500,0.750000,0.625000
6,Chris Whitney,2003,15,5,11:42,3.133333,0.866667,0.933333
7,Gary Payton,2003,82,82,34:35,14.621951,4.170732,5.475610
8,Gary Payton,2004,77,77,33:08,11.337662,3.064935,6.090909
9,Gary Payton,2005,81,25,28:27,7.728395,2.876543,3.172840
10,Gary Payton,2006,68,28,22:06,5.264706,1.941176,2.955882


In [58]:
print(moyennes.isna().sum())

PLAYER_ID         0
SEASON            0
PTS               0
REB               0
AST               0
STL               0
BLK               0
FGA               0
FGM               0
FG3A              0
FG3M              0
FTA               0
FTM               0
OREB              0
DREB              0
TO                0
PF                0
FG_PCT           44
FG3_PCT        1185
FT_PCT          404
MIN_DEC           0
PLAYER_NAME       0
GP                0
GS                0
MIN_MOY           0
dtype: int64


In [59]:
colonnes_pct = ["FG_PCT", "FG3_PCT", "FT_PCT"]

moyennes[colonnes_pct] = moyennes[colonnes_pct].fillna(0)

print(moyennes.isna().sum())

PLAYER_ID      0
SEASON         0
PTS            0
REB            0
AST            0
STL            0
BLK            0
FGA            0
FGM            0
FG3A           0
FG3M           0
FTA            0
FTM            0
OREB           0
DREB           0
TO             0
PF             0
FG_PCT         0
FG3_PCT        0
FT_PCT         0
MIN_DEC        0
PLAYER_NAME    0
GP             0
GS             0
MIN_MOY        0
dtype: int64


In [60]:
print("Nombre de joueurs-saisons :", len(moyennes))
print("Nombre de colonnes :", len(moyennes.columns))

print("\nListe des colonnes :")
print(moyennes.columns.tolist())

Nombre de joueurs-saisons : 9728
Nombre de colonnes : 25

Liste des colonnes :
['PLAYER_ID', 'SEASON', 'PTS', 'REB', 'AST', 'STL', 'BLK', 'FGA', 'FGM', 'FG3A', 'FG3M', 'FTA', 'FTM', 'OREB', 'DREB', 'TO', 'PF', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'MIN_DEC', 'PLAYER_NAME', 'GP', 'GS', 'MIN_MOY']


In [61]:
equipes = data_reg.groupby(
    ["PLAYER_ID", "SEASON"]
)["TEAM_ABBREVIATION"].nunique().reset_index()

equipes = equipes.rename(
    columns={"TEAM_ABBREVIATION": "NB_EQUIPES"}
)

print(equipes["NB_EQUIPES"].value_counts().sort_index())

NB_EQUIPES
1    8585
2    1149
3      98
4       3
5       1
Name: count, dtype: int64


In [62]:
equipes_joueurs = data_reg.groupby(
    ["PLAYER_ID", "SEASON"]
)["TEAM_ABBREVIATION"].unique().reset_index()

display(equipes_joueurs.head(10))

,PLAYER_ID,SEASON,TEAM_ABBREVIATION
0,15,2003,[HOU]
1,15,2004,[CHI]
2,15,2005,[CHI]
3,15,2006,[PHX]
4,15,2007,[PHX]
5,42,2003,[ORL]
6,43,2003,[WAS]
7,56,2003,[LAL]
8,56,2004,"[BOS, ATL]"
9,56,2005,[MIA]


In [63]:
equipes_joueurs["TEAM_ABBREVIATION"] = equipes_joueurs["TEAM_ABBREVIATION"].apply(
    lambda x: ", ".join(x)
)

moyennes = moyennes.merge(
    equipes_joueurs,
    on=["PLAYER_ID", "SEASON"],
    how="left"
)

display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "TEAM_ABBREVIATION", "GP", "GS", "PTS"]
    ].head(10)
)

,PLAYER_NAME,SEASON,TEAM_ABBREVIATION,GP,GS,PTS
0,Eric Piatkowski,2003,HOU,49,0,4.102041
1,Eric Piatkowski,2004,CHI,68,11,4.764706
2,Eric Piatkowski,2005,CHI,29,1,2.034483
3,Eric Piatkowski,2006,PHX,11,0,2.454545
4,Eric Piatkowski,2007,PHX,16,0,2.437500
5,Chris Whitney,2003,WAS,15,5,3.133333
6,Gary Payton,2003,LAL,82,82,14.621951
7,Gary Payton,2004,"BOS, ATL",77,77,11.337662
8,Gary Payton,2005,MIA,81,25,7.728395
9,Gary Payton,2006,MIA,68,28,5.264706


In [64]:
print("Nombre de lignes :", len(moyennes))

print("Doublons joueurs-saisons :", moyennes.duplicated(
    subset=["PLAYER_ID", "SEASON"]
).sum())

print("Équipes manquantes :", moyennes["TEAM_ABBREVIATION"].isna().sum())

Nombre de lignes : 9728
Doublons joueurs-saisons : 0
Équipes manquantes : 0


In [66]:
colonnes = moyennes.columns.tolist()

colonnes.remove("PLAYER_NAME")
colonnes.insert(1, "PLAYER_NAME")

moyennes = moyennes[colonnes]

display(moyennes.head())

,PLAYER_ID,PLAYER_NAME,SEASON,PTS,REB,AST,STL,BLK,FGA,FGM,...,TO,PF,FG_PCT,FG3_PCT,FT_PCT,MIN_DEC,GP,GS,MIN_MOY,TEAM_ABBREVIATION
0,15,Eric Piatkowski,2003,4.102041,1.489796,0.530612,0.326531,0.102041,3.897959,1.469388,...,0.591837,0.857143,37.696335,35.245902,87.500000,14.328231,49,0,14:20,HOU
1,15,Eric Piatkowski,2004,4.764706,1.161765,0.750000,0.426471,0.014706,3.794118,1.632353,...,0.544118,0.985294,43.023256,42.537313,80.357143,12.340931,68,11,12:20,CHI
2,15,Eric Piatkowski,2005,2.034483,0.793103,0.448276,0.206897,0.034483,2.103448,0.827586,...,0.379310,0.689655,39.344262,27.272727,40.000000,7.870115,29,1,07:52,CHI
3,15,Eric Piatkowski,2006,2.454545,0.818182,0.363636,0.000000,0.090909,2.272727,0.818182,...,0.454545,0.545455,36.000000,38.888889,100.000000,6.618182,11,0,06:37,PHX
4,15,Eric Piatkowski,2007,2.437500,0.750000,0.625000,0.000000,0.062500,2.062500,0.750000,...,0.375000,0.687500,36.363636,42.307692,100.000000,7.041667,16,0,07:02,PHX


In [67]:
pd.options.display.float_format = "{:.2f}".format

display(moyennes.head(10))

,PLAYER_ID,PLAYER_NAME,SEASON,PTS,REB,AST,STL,BLK,FGA,FGM,...,TO,PF,FG_PCT,FG3_PCT,FT_PCT,MIN_DEC,GP,GS,MIN_MOY,TEAM_ABBREVIATION
0,15,Eric Piatkowski,2003,4.10,1.49,0.53,0.33,0.10,3.90,1.47,...,0.59,0.86,37.70,35.25,87.50,14.33,49,0,14:20,HOU
1,15,Eric Piatkowski,2004,4.76,1.16,0.75,0.43,0.01,3.79,1.63,...,0.54,0.99,43.02,42.54,80.36,12.34,68,11,12:20,CHI
2,15,Eric Piatkowski,2005,2.03,0.79,0.45,0.21,0.03,2.10,0.83,...,0.38,0.69,39.34,27.27,40.00,7.87,29,1,07:52,CHI
3,15,Eric Piatkowski,2006,2.45,0.82,0.36,0.00,0.09,2.27,0.82,...,0.45,0.55,36.00,38.89,100.00,6.62,11,0,06:37,PHX
4,15,Eric Piatkowski,2007,2.44,0.75,0.62,0.00,0.06,2.06,0.75,...,0.38,0.69,36.36,42.31,100.00,7.04,16,0,07:02,PHX
5,43,Chris Whitney,2003,3.13,0.87,0.93,0.40,0.13,2.80,1.13,...,0.40,0.40,40.48,47.06,100.00,11.71,15,5,11:42,WAS
6,56,Gary Payton,2003,14.62,4.17,5.48,1.17,0.23,12.49,5.88,...,1.84,2.06,47.07,33.33,71.43,34.59,82,82,34:35,LAL
7,56,Gary Payton,2004,11.34,3.06,6.09,1.14,0.16,9.42,4.40,...,1.92,1.57,46.76,32.56,76.12,33.13,77,77,33:08,"BOS, ATL"
8,56,Gary Payton,2005,7.73,2.88,3.17,0.88,0.12,6.75,2.84,...,1.26,2.09,42.05,28.70,79.37,28.46,81,25,28:27,MIA
9,56,Gary Payton,2006,5.26,1.94,2.96,0.63,0.04,5.38,2.12,...,0.97,1.72,39.34,26.02,66.67,22.10,68,28,22:06,MIA


In [68]:
moyennes.to_csv("nba_joueurs_saisons.csv", index=False)

In [69]:
moyennes.round(2).to_csv(
    "nba_joueurs_saisons_lisible.csv",
    index=False
)

# Partie 2 — Classification des joueurs : les 12 archétypes

In [70]:
print(data_reg["START_POSITION"].value_counts(dropna=False))

START_POSITION
NaN    345551
F       92975
G       92973
C       46486
Name: count, dtype: int64


In [71]:
colonnes_archetypes = [
    "PTS", "FGM", "FGA", "FG_PCT",
    "FG3M", "FG3A", "FG3_PCT",
    "FTM", "FTA", "FT_PCT",
    "REB", "OREB", "DREB",
    "AST", "TO", "STL", "BLK", "PF",
    "MIN_DEC", "GP", "GS"
]

display(moyennes[colonnes_archetypes].describe())

,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,...,OREB,DREB,AST,TO,STL,BLK,PF,MIN_DEC,GP,GS
count,9728.00,9728.00,9728.00,9728.00,9728.00,9728.00,9728.00,9728.00,9728.00,9728.00,...,9728.00,9728.00,9728.00,9728.00,9728.00,9728.00,9728.00,9728.00,9728.00,9728.00
mean,8.32,3.09,6.87,43.89,0.68,1.95,25.56,1.45,1.92,70.08,...,0.91,2.65,1.83,1.15,0.63,0.40,1.78,20.12,49.60,23.89
std,6.06,2.20,4.63,10.48,0.75,1.97,16.96,1.41,1.75,20.46,...,0.80,1.80,1.80,0.80,0.43,0.45,0.79,9.80,25.27,27.87
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.07,1.00,0.00
25%,3.67,1.38,3.26,40.00,0.01,0.16,10.00,0.50,0.73,64.65,...,0.34,1.35,0.59,0.57,0.32,0.11,1.22,12.15,28.00,1.00
50%,6.82,2.58,5.77,43.95,0.45,1.44,31.51,1.00,1.40,75.00,...,0.66,2.28,1.22,0.96,0.55,0.26,1.80,19.58,56.00,10.00
75%,11.65,4.34,9.53,48.57,1.10,3.15,36.91,1.91,2.56,82.12,...,1.23,3.49,2.41,1.54,0.86,0.52,2.33,28.19,72.00,45.00
max,36.13,12.22,27.16,100.00,5.35,13.18,100.00,10.27,12.62,100.00,...,5.52,11.37,11.74,5.73,2.89,3.68,6.00,43.52,85.00,83.00


In [72]:
stats_3D = ["FG3A", "FG3_PCT", "STL", "BLK"]

display(moyennes[stats_3D].describe())

,FG3A,FG3_PCT,STL,BLK
count,9728.00,9728.00,9728.00,9728.00
mean,1.95,25.56,0.63,0.40
std,1.97,16.96,0.43,0.45
min,0.00,0.00,0.00,0.00
25%,0.16,10.00,0.32,0.11
50%,1.44,31.51,0.55,0.26
75%,3.15,36.91,0.86,0.52
max,13.18,100.00,2.89,3.68


In [73]:
moyennes["PART_3PTS"] = (
    moyennes["FG3A"] / moyennes["FGA"] * 100
).fillna(0)

display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "FG3A", "FGA", "PART_3PTS"]
    ].head(10)
)

,PLAYER_NAME,SEASON,FG3A,FGA,PART_3PTS
0,Eric Piatkowski,2003,2.49,3.90,63.87
1,Eric Piatkowski,2004,1.97,3.79,51.94
2,Eric Piatkowski,2005,1.14,2.10,54.10
3,Eric Piatkowski,2006,1.64,2.27,72.00
4,Eric Piatkowski,2007,1.62,2.06,78.79
5,Chris Whitney,2003,1.13,2.80,40.48
6,Gary Payton,2003,2.01,12.49,16.11
7,Gary Payton,2004,1.68,9.42,17.79
8,Gary Payton,2005,2.84,6.75,42.05
9,Gary Payton,2006,1.81,5.38,33.61


In [74]:
print(moyennes["PART_3PTS"].describe())

count   9728.00
mean      27.63
std       22.48
min        0.00
25%        3.34
50%       27.83
75%       43.71
max      100.00
Name: PART_3PTS, dtype: float64


In [75]:
display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "PART_3PTS",
         "FG3A", "FG3_PCT", "STL", "BLK", "GP"]
    ].sort_values(
        by="PART_3PTS",
        ascending=False
    ).head(15)
)

,PLAYER_NAME,SEASON,PART_3PTS,FG3A,FG3_PCT,STL,BLK,GP
7376,James Michael McAdoo,2017,100.00,2.33,28.57,0.00,0.33,3
9514,Rob Edwards,2021,100.00,2.00,25.00,0.00,0.00,2
9496,Will Magnay,2020,100.00,1.00,0.00,0.00,0.00,1
9487,Marko Simonovic,2022,100.00,1.00,0.00,0.00,0.00,1
9481,Trevelin Queen,2022,100.00,1.50,0.00,0.50,0.50,2
9471,Anthony Lamb,2021,100.00,1.00,0.00,0.00,0.00,2
9297,Matt Mooney,2021,100.00,1.00,0.00,1.00,0.00,1
1875,Bobby Simmons,2010,100.00,1.50,0.00,0.00,0.00,2
1968,Dajuan Wagner,2006,100.00,1.00,100.00,0.00,0.00,1
9291,Dakota Mathias,2021,100.00,1.00,33.33,0.17,0.00,6


In [76]:
volume_tirs = data_reg.groupby(
    ["PLAYER_ID", "SEASON"]
)[["FGA", "FG3A"]].sum().reset_index()

volume_tirs = volume_tirs.rename(columns={
    "FGA": "FGA_TOTAL",
    "FG3A": "FG3A_TOTAL"
})

display(volume_tirs.head(10))

,PLAYER_ID,SEASON,FGA_TOTAL,FG3A_TOTAL
0,15,2003,191.00,122.00
1,15,2004,258.00,134.00
2,15,2005,61.00,33.00
3,15,2006,25.00,18.00
4,15,2007,33.00,26.00
5,42,2003,0.00,0.00
6,43,2003,42.00,17.00
7,56,2003,1024.00,165.00
8,56,2004,725.00,129.00
9,56,2005,547.00,230.00


In [77]:
moyennes = moyennes.merge(
    volume_tirs,
    on=["PLAYER_ID", "SEASON"],
    how="left"
)

display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "PART_3PTS",
         "FGA_TOTAL", "FG3A_TOTAL"]
    ].head(10)
)

,PLAYER_NAME,SEASON,PART_3PTS,FGA_TOTAL,FG3A_TOTAL
0,Eric Piatkowski,2003,63.87,191.00,122.00
1,Eric Piatkowski,2004,51.94,258.00,134.00
2,Eric Piatkowski,2005,54.10,61.00,33.00
3,Eric Piatkowski,2006,72.00,25.00,18.00
4,Eric Piatkowski,2007,78.79,33.00,26.00
5,Chris Whitney,2003,40.48,42.00,17.00
6,Gary Payton,2003,16.11,1024.00,165.00
7,Gary Payton,2004,17.79,725.00,129.00
8,Gary Payton,2005,42.05,547.00,230.00
9,Gary Payton,2006,33.61,366.00,123.00


In [78]:
moyennes["FIABILITE_TIRS"] = (
    moyennes["FGA_TOTAL"] /
    (moyennes["FGA_TOTAL"] + 100)
)

display(
    moyennes[
        ["PLAYER_NAME", "SEASON",
         "FGA_TOTAL", "PART_3PTS", "FIABILITE_TIRS"]
    ].head(10)
)

,PLAYER_NAME,SEASON,FGA_TOTAL,PART_3PTS,FIABILITE_TIRS
0,Eric Piatkowski,2003,191.00,63.87,0.66
1,Eric Piatkowski,2004,258.00,51.94,0.72
2,Eric Piatkowski,2005,61.00,54.10,0.38
3,Eric Piatkowski,2006,25.00,72.00,0.20
4,Eric Piatkowski,2007,33.00,78.79,0.25
5,Chris Whitney,2003,42.00,40.48,0.30
6,Gary Payton,2003,1024.00,16.11,0.91
7,Gary Payton,2004,725.00,17.79,0.88
8,Gary Payton,2005,547.00,42.05,0.85
9,Gary Payton,2006,366.00,33.61,0.79


In [79]:
moyenne_3pts = moyennes["PART_3PTS"].mean()

moyennes["PART_3PTS_AJUSTEE"] = (
    moyennes["FIABILITE_TIRS"] * moyennes["PART_3PTS"]
    + (1 - moyennes["FIABILITE_TIRS"]) * moyenne_3pts
)

display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "PART_3PTS",
         "FIABILITE_TIRS", "PART_3PTS_AJUSTEE"]
    ].head(10)
)

,PLAYER_NAME,SEASON,PART_3PTS,FIABILITE_TIRS,PART_3PTS_AJUSTEE
0,Eric Piatkowski,2003,63.87,0.66,51.42
1,Eric Piatkowski,2004,51.94,0.72,45.15
2,Eric Piatkowski,2005,54.10,0.38,37.66
3,Eric Piatkowski,2006,72.00,0.20,36.50
4,Eric Piatkowski,2007,78.79,0.25,40.32
5,Chris Whitney,2003,40.48,0.30,31.43
6,Gary Payton,2003,16.11,0.91,17.14
7,Gary Payton,2004,17.79,0.88,18.99
8,Gary Payton,2005,42.05,0.85,39.82
9,Gary Payton,2006,33.61,0.79,32.32


In [80]:
moyenne_fg3 = moyennes["FG3_PCT"].mean()

moyennes["FG3_PCT_AJUSTE"] = (
    moyennes["FG3A_TOTAL"] * moyennes["FG3_PCT"]
    + 100 * moyenne_fg3
) / (moyennes["FG3A_TOTAL"] + 100)

display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "FG3A_TOTAL",
         "FG3_PCT", "FG3_PCT_AJUSTE"]
    ].head(10)
)

,PLAYER_NAME,SEASON,FG3A_TOTAL,FG3_PCT,FG3_PCT_AJUSTE
0,Eric Piatkowski,2003,122.00,35.25,30.88
1,Eric Piatkowski,2004,134.00,42.54,35.28
2,Eric Piatkowski,2005,33.00,27.27,25.98
3,Eric Piatkowski,2006,18.00,38.89,27.59
4,Eric Piatkowski,2007,26.00,42.31,29.01
5,Chris Whitney,2003,17.00,47.06,28.68
6,Gary Payton,2003,165.00,33.33,30.40
7,Gary Payton,2004,129.00,32.56,29.50
8,Gary Payton,2005,230.00,28.70,27.74
9,Gary Payton,2006,123.00,26.02,25.81


In [81]:
display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "GP",
         "MIN_DEC", "STL", "BLK"]
    ].sort_values(
        by="STL",
        ascending=False
    ).head(15)
)

,PLAYER_NAME,SEASON,GP,MIN_DEC,STL,BLK
821,Larry Hughes,2004,61,38.65,2.89,0.30
3251,Chris Paul,2008,78,38.49,2.77,0.13
7606,Delon Wright,2022,4,22.69,2.75,1.00
3250,Chris Paul,2007,80,37.57,2.71,0.05
3254,Chris Paul,2011,60,36.34,2.53,0.07
1749,Gerald Wallace,2005,55,34.46,2.51,2.09
3256,Chris Paul,2013,62,35.02,2.48,0.06
3255,Chris Paul,2012,70,33.36,2.41,0.14
5104,Ricky Rubio,2012,57,29.66,2.40,0.09
373,Allen Iverson,2004,75,42.35,2.40,0.12


In [82]:
moyennes["STL_MIN"] = moyennes["STL"] / moyennes["MIN_DEC"]
moyennes["BLK_MIN"] = moyennes["BLK"] / moyennes["MIN_DEC"]

display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "GP",
         "MIN_DEC", "STL", "STL_MIN",
         "BLK", "BLK_MIN"]
    ].head(10)
)

,PLAYER_NAME,SEASON,GP,MIN_DEC,STL,STL_MIN,BLK,BLK_MIN
0,Eric Piatkowski,2003,49,14.33,0.33,0.02,0.10,0.01
1,Eric Piatkowski,2004,68,12.34,0.43,0.03,0.01,0.00
2,Eric Piatkowski,2005,29,7.87,0.21,0.03,0.03,0.00
3,Eric Piatkowski,2006,11,6.62,0.00,0.00,0.09,0.01
4,Eric Piatkowski,2007,16,7.04,0.00,0.00,0.06,0.01
5,Chris Whitney,2003,15,11.71,0.40,0.03,0.13,0.01
6,Gary Payton,2003,82,34.59,1.17,0.03,0.23,0.01
7,Gary Payton,2004,77,33.13,1.14,0.03,0.16,0.00
8,Gary Payton,2005,81,28.46,0.88,0.03,0.12,0.00
9,Gary Payton,2006,68,22.10,0.63,0.03,0.04,0.00


In [83]:
moyennes["MIN_TOTAL"] = moyennes["MIN_DEC"] * moyennes["GP"]

display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "GP",
         "MIN_DEC", "MIN_TOTAL"]
    ].head(10)
)

,PLAYER_NAME,SEASON,GP,MIN_DEC,MIN_TOTAL
0,Eric Piatkowski,2003,49,14.33,702.08
1,Eric Piatkowski,2004,68,12.34,839.18
2,Eric Piatkowski,2005,29,7.87,228.23
3,Eric Piatkowski,2006,11,6.62,72.80
4,Eric Piatkowski,2007,16,7.04,112.67
5,Chris Whitney,2003,15,11.71,175.60
6,Gary Payton,2003,82,34.59,2836.07
7,Gary Payton,2004,77,33.13,2550.65
8,Gary Payton,2005,81,28.46,2304.92
9,Gary Payton,2006,68,22.10,1502.80


In [84]:
moyennes["FIABILITE_DEF"] = (
    moyennes["MIN_TOTAL"] /
    (moyennes["MIN_TOTAL"] + 500)
)

display(
    moyennes[
        ["PLAYER_NAME", "SEASON",
         "MIN_TOTAL", "FIABILITE_DEF"]
    ].head(10)
)

,PLAYER_NAME,SEASON,MIN_TOTAL,FIABILITE_DEF
0,Eric Piatkowski,2003,702.08,0.58
1,Eric Piatkowski,2004,839.18,0.63
2,Eric Piatkowski,2005,228.23,0.31
3,Eric Piatkowski,2006,72.80,0.13
4,Eric Piatkowski,2007,112.67,0.18
5,Chris Whitney,2003,175.60,0.26
6,Gary Payton,2003,2836.07,0.85
7,Gary Payton,2004,2550.65,0.84
8,Gary Payton,2005,2304.92,0.82
9,Gary Payton,2006,1502.80,0.75


In [85]:
moyenne_stl = moyennes["STL"].sum() / moyennes["MIN_DEC"].sum()
moyenne_blk = moyennes["BLK"].sum() / moyennes["MIN_DEC"].sum()

moyennes["STL_MIN_AJUSTE"] = (
    moyennes["FIABILITE_DEF"] * moyennes["STL_MIN"]
    + (1 - moyennes["FIABILITE_DEF"]) * moyenne_stl
)

moyennes["BLK_MIN_AJUSTE"] = (
    moyennes["FIABILITE_DEF"] * moyennes["BLK_MIN"]
    + (1 - moyennes["FIABILITE_DEF"]) * moyenne_blk
)

display(
    moyennes[
        ["PLAYER_NAME", "SEASON",
         "STL_MIN", "STL_MIN_AJUSTE",
         "BLK_MIN", "BLK_MIN_AJUSTE"]
    ].head(10)
)

,PLAYER_NAME,SEASON,STL_MIN,STL_MIN_AJUSTE,BLK_MIN,BLK_MIN_AJUSTE
0,Eric Piatkowski,2003,0.02,0.03,0.01,0.01
1,Eric Piatkowski,2004,0.03,0.03,0.00,0.01
2,Eric Piatkowski,2005,0.03,0.03,0.00,0.02
3,Eric Piatkowski,2006,0.00,0.03,0.01,0.02
4,Eric Piatkowski,2007,0.00,0.03,0.01,0.02
5,Chris Whitney,2003,0.03,0.03,0.01,0.02
6,Gary Payton,2003,0.03,0.03,0.01,0.01
7,Gary Payton,2004,0.03,0.03,0.00,0.01
8,Gary Payton,2005,0.03,0.03,0.00,0.01
9,Gary Payton,2006,0.03,0.03,0.00,0.01


In [86]:
moyennes["STL_100_AJUSTE"] = moyennes["STL_MIN_AJUSTE"] * 100
moyennes["BLK_100_AJUSTE"] = moyennes["BLK_MIN_AJUSTE"] * 100

display(
    moyennes[
        ["PLAYER_NAME", "SEASON",
         "STL_100_AJUSTE", "BLK_100_AJUSTE"]
    ].head(10)
)

,PLAYER_NAME,SEASON,STL_100_AJUSTE,BLK_100_AJUSTE
0,Eric Piatkowski,2003,2.63,1.25
1,Eric Piatkowski,2004,3.33,0.82
2,Eric Piatkowski,2005,2.96,1.51
3,Eric Piatkowski,2006,2.72,1.92
4,Eric Piatkowski,2007,2.54,1.79
5,Chris Whitney,2003,3.19,1.78
6,Gary Payton,2003,3.34,0.87
7,Gary Payton,2004,3.39,0.72
8,Gary Payton,2005,3.09,0.71
9,Gary Payton,2006,2.92,0.65


In [87]:
stats_3D = [
    "PART_3PTS_AJUSTEE",
    "FG3_PCT_AJUSTE",
    "STL_100_AJUSTE",
    "BLK_100_AJUSTE"
]

for stat in stats_3D:
    moyennes[stat + "_SCORE"] = (
        moyennes[stat].rank(pct=True) * 100
    )

display(
    moyennes[
        ["PLAYER_NAME", "SEASON",
         "PART_3PTS_AJUSTEE_SCORE",
         "FG3_PCT_AJUSTE_SCORE",
         "STL_100_AJUSTE_SCORE",
         "BLK_100_AJUSTE_SCORE"]
    ].head(10)
)

,PLAYER_NAME,SEASON,PART_3PTS_AJUSTEE_SCORE,FG3_PCT_AJUSTE_SCORE,STL_100_AJUSTE_SCORE,BLK_100_AJUSTE_SCORE
0,Eric Piatkowski,2003,94.50,69.94,25.39,24.58
1,Eric Piatkowski,2004,87.85,91.52,69.97,7.60
2,Eric Piatkowski,2005,75.09,39.65,45.45,36.66
3,Eric Piatkowski,2006,72.58,51.94,30.43,57.93
4,Eric Piatkowski,2007,80.43,59.69,21.33,50.43
5,Chris Whitney,2003,61.05,58.03,61.90,49.47
6,Gary Payton,2003,25.23,67.33,70.75,9.26
7,Gary Payton,2004,28.23,62.53,72.81,4.80
8,Gary Payton,2005,79.38,52.74,54.88,4.55
9,Gary Payton,2006,63.30,38.53,42.87,2.96


In [88]:
moyennes["SCORE_3D"] = (
    0.25 * moyennes["PART_3PTS_AJUSTEE_SCORE"]
    + 0.25 * moyennes["FG3_PCT_AJUSTE_SCORE"]
    + 0.35 * moyennes["STL_100_AJUSTE_SCORE"]
    + 0.15 * moyennes["BLK_100_AJUSTE_SCORE"]
)

display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "SCORE_3D"]
    ].sort_values(
        by="SCORE_3D",
        ascending=False
    ).head(15)
)

,PLAYER_NAME,SEASON,SCORE_3D
6942,Robert Covington,2021,95.23
5424,Danny Green,2020,94.58
5418,Danny Green,2014,94.30
6938,Robert Covington,2017,93.92
8277,Lonzo Ball,2021,93.89
6939,Robert Covington,2018,93.42
6941,Robert Covington,2020,93.28
6699,Justin Holiday,2019,93.15
8748,De'Anthony Melton,2020,92.69
6940,Robert Covington,2019,91.91


In [89]:
moyennes["FG3A_SCORE"] = moyennes["FG3A"].rank(pct=True) * 100

display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "FG3A",
         "FG3A_SCORE", "SCORE_3D"]
    ].sort_values(
        by="SCORE_3D",
        ascending=False
    ).head(15)
)

,PLAYER_NAME,SEASON,FG3A,FG3A_SCORE,SCORE_3D
6942,Robert Covington,2021,4.77,90.10,95.23
5424,Danny Green,2020,6.26,96.50,94.58
5418,Danny Green,2014,5.64,94.68,94.30
6938,Robert Covington,2017,6.88,97.82,93.92
8277,Lonzo Ball,2021,7.43,98.63,93.89
6939,Robert Covington,2018,6.43,96.94,93.42
6941,Robert Covington,2020,5.09,91.96,93.28
6699,Justin Holiday,2019,4.53,88.50,93.15
8748,De'Anthony Melton,2020,4.06,84.72,92.69
6940,Robert Covington,2019,6.86,97.79,91.91


In [92]:
moyennes["SCORE_3D"] = (
    0.20 * moyennes["PART_3PTS_AJUSTEE_SCORE"]
    + 0.20 * moyennes["FG3_PCT_AJUSTE_SCORE"]
    + 0.10 * moyennes["FG3A_SCORE"]
    + 0.35 * moyennes["STL_100_AJUSTE_SCORE"]
    + 0.15 * moyennes["BLK_100_AJUSTE_SCORE"]
)

display(
    moyennes[
        ["PLAYER_NAME", "SEASON", "SCORE_3D"]
    ].sort_values(
        by="SCORE_3D",
        ascending=False
    ).head(20)
)

,PLAYER_NAME,SEASON,SCORE_3D
6942,Robert Covington,2021,94.80
5424,Danny Green,2020,94.36
6938,Robert Covington,2017,94.22
6939,Robert Covington,2018,94.00
8277,Lonzo Ball,2021,93.97
5418,Danny Green,2014,93.96
6941,Robert Covington,2020,92.96
6940,Robert Covington,2019,92.91
6699,Justin Holiday,2019,92.26
5088,James Harden,2018,91.87
